In [1]:
import os
import sys
os.chdir("..")
sys.path.append(os.getcwd())
import pandas as pd
import duckdb
from pathlib import Path
from config import RESEARCH_ROOT, NSE_DB_PATH

In [2]:
RESEARCH_DB_PATH = RESEARCH_ROOT / "research.db"
con = duckdb.connect(RESEARCH_DB_PATH)
con.execute(f"ATTACH '{NSE_DB_PATH}' AS nse (READ_ONLY)") 

In [3]:
print(f"Connected to DuckDB: {RESEARCH_DB_PATH.resolve()}")

Connected to DuckDB: E:\Projects\embR\data\research.db


In [25]:
# read from nse, write to research.db
con.execute("""
    CREATE OR REPLACE TABLE forward_returns AS
    WITH daily AS (
        SELECT
            trade_date,
            index_name,
            close
        FROM nse.market_activity_index
        WHERE index_name = 'NIFTY 50'   -- adjust to exact name in your data
        ORDER BY trade_date
    )
    SELECT
        d.trade_date,
        d.index_name,
        d.close,

        -- forward returns
        ROUND((f1.close - d.close) / d.close * 100, 4) AS fwd_ret_1d,
        ROUND((f5.close - d.close) / d.close * 100, 4) AS fwd_ret_5d,
        ROUND((f20.close - d.close) / d.close * 100, 4) AS fwd_ret_20d,

        -- direction label (for classification)
        CASE WHEN f1.close > d.close THEN 1 ELSE 0 END AS up_1d

    FROM daily d
    LEFT JOIN daily f1
        ON f1.trade_date = (
            SELECT MIN(trade_date) FROM daily
            WHERE trade_date > d.trade_date
        )
    LEFT JOIN daily f5
        ON f5.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 4
        )
    LEFT JOIN daily f20
        ON f20.trade_date = (
            SELECT trade_date FROM daily
            WHERE trade_date > d.trade_date
            ORDER BY trade_date LIMIT 1 OFFSET 19
        )
    ;
""")

In [26]:
results = con.execute(""" SELECT * FROM forward_returns ORDER BY trade_date ASC LIMIT 10 OFFSET 20; """).fetchall()
for row in results:
    print(row)

(datetime.date(2016, 2, 1), 'NIFTY 50', 7555.95, -1.3288, -2.2327, -7.5292, 0)
(datetime.date(2016, 2, 2), 'NIFTY 50', 7455.55, -1.2575, -2.1105, -3.1285, 0)
(datetime.date(2016, 2, 3), 'NIFTY 50', 7361.8, 0.5732, -1.9846, 0.0958, 1)
(datetime.date(2016, 2, 4), 'NIFTY 50', 7404.0, 1.1494, -5.7759, 0.967, 1)
(datetime.date(2016, 2, 5), 'NIFTY 50', 7489.1, -1.36, -6.7852, -0.0501, 0)
(datetime.date(2016, 2, 8), 'NIFTY 50', 7387.25, -1.2055, -3.0363, 1.3273, 0)
(datetime.date(2016, 2, 9), 'NIFTY 50', 7298.2, -1.1304, -3.4248, 3.2008, 0)
(datetime.date(2016, 2, 10), 'NIFTY 50', 7215.7, -3.3171, -1.4863, 3.7481, 0)
(datetime.date(2016, 2, 11), 'NIFTY 50', 6976.35, 0.0659, 3.0876, 7.6523, 1)
(datetime.date(2016, 2, 12), 'NIFTY 50', 6980.95, 2.6071, 3.2918, 7.9903, 1)


In [27]:
results = con.execute(""" SELECT DISTINCT index_name FROM nse.market_activity_index ORDER BY 1; """).fetchall()
for row in results:
    print(row)

('BHARATBOND-APR25',)
('BHARATBOND-APR30',)
('BHARATBOND-APR31',)
('BHARATBOND-APR32',)
('BHARATBOND-APR33',)
('INDIA VIX',)
('NIFTY 100',)
('NIFTY 200',)
('NIFTY 50',)
('NIFTY 500',)
('NIFTY ALPHA 50',)
('NIFTY ALPHALOWVOL',)
('NIFTY AQL 30',)
('NIFTY AQLV 30',)
('NIFTY AUTO',)
('NIFTY BANK',)
('NIFTY CAPITAL MKT',)
('NIFTY CEMENT',)
('NIFTY CHEMICALS',)
('NIFTY COMMODITIES',)
('NIFTY CONSR DURBL',)
('NIFTY CONSUMPTION',)
('NIFTY COREHOUSING',)
('NIFTY CORP MAATR',)
('NIFTY CPSE',)
('NIFTY DIV OPPS 50',)
('NIFTY ENERGY',)
('NIFTY EV',)
('NIFTY FIN SERVICE',)
('NIFTY FINSEREXBNK',)
('NIFTY FINSRV25 50',)
('NIFTY FMCG',)
('NIFTY FPI 150',)
('NIFTY GROWSECT 15',)
('NIFTY GS 10YR',)
('NIFTY GS 10YR CLN',)
('NIFTY GS 11 15YR',)
('NIFTY GS 15YRPLUS',)
('NIFTY GS 4 8YR',)
('NIFTY GS 8 13YR',)
('NIFTY GS COMPSITE',)
('NIFTY HEALTHCARE',)
('NIFTY HIGHBETA 50',)
('NIFTY HOUSING',)
('NIFTY IND DEFENCE',)
('NIFTY IND DIGITAL',)
('NIFTY IND TOURISM',)
('NIFTY INDIA MFG',)
('NIFTY INFRA',)
('NIFTY 

In [29]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close         AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close        AS vix_close

FROM forward_returns fr
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'INDIA VIX'

ORDER BY fr.trade_date
""")

In [31]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10 OFFSET 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 25), 24056.0, -0.4562, 0.8931, -1.1995, 0, 13.05)
(datetime.date(2026, 6, 24), 24021.65, 0.143, 0.6413, -0.633, 1, 13.385)
(datetime.date(2026, 6, 23), 23824.1, 0.8292, 0.7629, 0.7226, 1, 13.9425)
(datetime.date(2026, 6, 22), 24102.9, -1.1567, -0.9839, 0.3518, 0, 12.8425)
(datetime.date(2026, 6, 19), 24013.1, 0.374, -0.2784, 0.9387, 1, 12.97)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, -0.4634, 0.6881, 0, 12.6725)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, -0.2659, -0.0538, 1, 13.1875)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, -0.688, 0.3725, 1, 13.3625)
(datetime.date(2026, 6, 15), 23853.9, 0.567, 1.0439, 0.8307, 1, 14.3525)
(datetime.date(2026, 6, 12), 23622.9, 0.9779, 1.6518, 2.4895, 1, 14.7175)


In [32]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 13 THEN '1_low <13'
            WHEN vix_close < 16 THEN '2_calm 13-16'
            WHEN vix_close < 20 THEN '3_normal 16-20'
            WHEN vix_close < 25 THEN '4_elevated 20-25'
            ELSE                     '5_fear >25'
        END AS vix_regime,

        COUNT(*)                        AS days,
        ROUND(AVG(fwd_ret_1d), 3)       AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)       AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)      AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)      AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL

    GROUP BY 1
    ORDER BY 1
""").fetchall()

for row in results: print(row)

('1_low <13', 650, 0.053, 0.165, 1.037, 54.9)
('2_calm 13-16', 886, 0.014, 0.165, 0.178, 52.8)
('3_normal 16-20', 626, 0.012, 0.147, 0.607, 54.3)
('4_elevated 20-25', 320, 0.114, 0.441, 2.291, 53.1)
('5_fear >25', 132, 0.242, 1.102, 5.171, 60.6)


In [33]:
results = con.execute("""
    SELECT *
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
    ORDER BY trade_date DESC, expiry ASC
    LIMIT 20
""").fetchall()
for row in results: print(row)

('IDO', 'NIFTY', datetime.date(2026, 7, 28), datetime.date(2026, 7, 24), 23767.45000000009, 166311470.0, 201567860.0, 0.8250892280148234, 23900.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 4), datetime.date(2026, 7, 24), 23767.45000000008, 21752640.0, 28020070.0, 0.7763235423751618, 23900.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 11), datetime.date(2026, 7, 24), 23767.45000000008, 2481180.0, 3155945.0, 0.7861924082960888, 24000.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 18), datetime.date(2026, 7, 24), 23767.45000000007, 495300.0, 587860.0, 0.8425475453339231, 23900.0)
('IDO', 'NIFTY', datetime.date(2026, 8, 25), datetime.date(2026, 7, 24), 23767.45000000009, 35538490.0, 33887555.0, 1.048718032327797, 24050.0)
('IDO', 'NIFTY', datetime.date(2026, 9, 1), datetime.date(2026, 7, 24), 23767.450000000063, 32045.0, 17615.0, 1.8191881918819188, 24100.0)
('IDO', 'NIFTY', datetime.date(2026, 9, 29), datetime.date(2026, 7, 24), 23767.45000000009, 17164780.0, 11352285.0, 1.5120110180461466, 24500.

In [34]:
results = con.execute("""
    SELECT
        expiry,
        trade_date,
        pe_oi,
        ce_oi,
        pe_oi + ce_oi AS total_oi
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND trade_date = '2026-06-10'
    ORDER BY total_oi DESC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 16), datetime.date(2026, 6, 10), 93749955.0, 114460970.0, 208210925.0)
(datetime.date(2026, 6, 30), datetime.date(2026, 6, 10), 69249000.0, 66301370.0, 135550370.0)
(datetime.date(2026, 12, 29), datetime.date(2026, 6, 10), 11812480.0, 9371595.0, 21184075.0)
(datetime.date(2026, 6, 23), datetime.date(2026, 6, 10), 9232405.0, 10127780.0, 19360185.0)
(datetime.date(2026, 7, 28), datetime.date(2026, 6, 10), 9825335.0, 8697390.0, 18522725.0)
(datetime.date(2026, 9, 29), datetime.date(2026, 6, 10), 5249010.0, 4500755.0, 9749765.0)
(datetime.date(2026, 8, 25), datetime.date(2026, 6, 10), 2054260.0, 1737060.0, 3791320.0)
(datetime.date(2026, 7, 7), datetime.date(2026, 6, 10), 275860.0, 505245.0, 781105.0)
(datetime.date(2027, 12, 28), datetime.date(2026, 6, 10), 218430.0, 155535.0, 373965.0)
(datetime.date(2028, 12, 26), datetime.date(2026, 6, 10), 44980.0, 16115.0, 61095.0)
(datetime.date(2026, 7, 14), datetime.date(2026, 6, 10), 9425.0, 12155.0, 21580.0)
(datetime.dat

In [35]:
results = con.execute("""
    SELECT
        percentile_cont(0.25) WITHIN GROUP (ORDER BY total_oi) AS p25,
        percentile_cont(0.50) WITHIN GROUP (ORDER BY total_oi) AS p50,
        percentile_cont(0.75) WITHIN GROUP (ORDER BY total_oi) AS p75,
        percentile_cont(0.90) WITHIN GROUP (ORDER BY total_oi) AS p90,
        percentile_cont(0.95) WITHIN GROUP (ORDER BY total_oi) AS p95,
        MIN(total_oi)  AS min_oi,
        MAX(total_oi)  AS max_oi,
        COUNT(*)       AS total_rows
    FROM (
        SELECT pe_oi + ce_oi AS total_oi
        FROM nse.options_analytics
        WHERE ticker = 'NIFTY'
          AND pe_oi IS NOT NULL
          AND ce_oi IS NOT NULL
    )
""").fetchall()
for row in results: print(row)

(1600.0, 435475.0, 5764900.0, 34831700.0, 78583362.5, 0.0, 424741950.0, 46851)


In [36]:
results = con.execute("""
    SELECT
        trade_date,
        COUNT(*) AS liquid_expiries
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND (pe_oi + ce_oi) > 11800000
    GROUP BY trade_date
    ORDER BY trade_date DESC
    LIMIT 20
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 24), 5)
(datetime.date(2026, 7, 23), 5)
(datetime.date(2026, 7, 22), 5)
(datetime.date(2026, 7, 21), 6)
(datetime.date(2026, 7, 20), 5)
(datetime.date(2026, 7, 17), 5)
(datetime.date(2026, 7, 16), 5)
(datetime.date(2026, 7, 15), 5)
(datetime.date(2026, 7, 14), 6)
(datetime.date(2026, 7, 13), 6)
(datetime.date(2026, 7, 10), 6)
(datetime.date(2026, 7, 9), 6)
(datetime.date(2026, 7, 8), 6)
(datetime.date(2026, 7, 7), 6)
(datetime.date(2026, 7, 6), 6)
(datetime.date(2026, 7, 3), 6)
(datetime.date(2026, 7, 2), 6)
(datetime.date(2026, 7, 1), 6)
(datetime.date(2026, 6, 30), 6)
(datetime.date(2026, 6, 29), 5)


In [39]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close              AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,
    vix.close             AS vix_close,
    pcr_agg.pcr,
    mp_agg.max_pain_dist_pct

FROM forward_returns fr

-- VIX
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'INDIA VIX'

-- FO volatility
LEFT JOIN nse.fo_volatility fv
    ON fv.trade_date = fr.trade_date
    AND fv.ticker = 'NIFTY'

-- Market breadth
LEFT JOIN nse.market_activity_breadth mab
    ON mab.trade_date = fr.trade_date

-- Market activity summary
LEFT JOIN nse.market_activity_summary mas
    ON mas.trade_date = fr.trade_date

-- PCR: all expiries
LEFT JOIN (
    SELECT
        trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: liquid expiries only, OI-weighted
LEFT JOIN (
    SELECT oa.trade_date,
        ROUND(
            SUM(((fr2.close - oa.max_pain) / NULLIF(oa.max_pain, 0) * 100) * (oa.pe_oi + oa.ce_oi))
            / NULLIF(SUM(oa.pe_oi + oa.ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM nse.options_analytics oa
    JOIN forward_returns fr2 ON fr2.trade_date = oa.trade_date
    WHERE oa.ticker = 'NIFTY' AND oa.instrument_type = 'IDO'
      AND (oa.pe_oi + oa.ce_oi) > 11800000
    GROUP BY oa.trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [42]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10
    OFFSET 20;
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 25), 24056.0, -0.4562, 0.8931, -1.1995, 0, 13.05, 1.0608, -0.5301)
(datetime.date(2026, 6, 24), 24021.65, 0.143, 0.6413, -0.633, 1, 13.385, 1.2085, -0.3082)
(datetime.date(2026, 6, 23), 23824.1, 0.8292, 0.7629, 0.7226, 1, 13.9425, 0.7193, -0.6458)
(datetime.date(2026, 6, 22), 24102.9, -1.1567, -0.9839, 0.3518, 0, 12.8425, 0.9654, -0.2197)
(datetime.date(2026, 6, 19), 24013.1, 0.374, -0.2784, 0.9387, 1, 12.97, 0.91, -0.1318)
(datetime.date(2026, 6, 18), 24168.0, -0.6409, -0.4634, 0.6881, 0, 12.6725, 1.1229, 0.2006)
(datetime.date(2026, 6, 17), 24085.7, 0.3417, -0.2659, -0.0538, 1, 13.1875, 1.1, 0.1268)
(datetime.date(2026, 6, 16), 23989.15, 0.4025, -0.688, 0.3725, 1, 13.3625, 1.1117, 0.0237)
(datetime.date(2026, 6, 15), 23853.9, 0.567, 1.0439, 0.8307, 1, 14.3525, 0.9881, -0.1728)
(datetime.date(2026, 6, 12), 23622.9, 0.9779, 1.6518, 2.4895, 1, 14.7175, 1.412, -0.3175)


In [43]:
results = con.execute("""
    SELECT
        CASE
            WHEN pcr < 0.7  THEN '1_very_low <0.7'
            WHEN pcr < 0.9  THEN '2_low 0.7-0.9'
            WHEN pcr < 1.1  THEN '3_neutral 0.9-1.1'
            WHEN pcr < 1.3  THEN '4_high 1.1-1.3'
            ELSE                 '5_very_high >1.3'
        END AS pcr_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND pcr IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== PCR ===")
for row in results: print(row)

results = con.execute("""
    SELECT
        CASE
            WHEN max_pain_dist_pct < -3   THEN '1_far_below <-3%'
            WHEN max_pain_dist_pct < -1.5 THEN '2_below -3 to -1.5%'
            WHEN max_pain_dist_pct < 0    THEN '3_slightly_below -1.5 to 0%'
            WHEN max_pain_dist_pct < 1.5  THEN '4_slightly_above 0 to 1.5%'
            ELSE                               '5_far_above >1.5%'
        END AS mp_bucket,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL AND max_pain_dist_pct IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Max Pain Distance ===")
for row in results: print(row)

=== PCR ===
('1_very_low <0.7', 55, 0.025, 0.577, 0.764, 52.7)
('2_low 0.7-0.9', 459, 0.032, 0.156, 0.613, 48.4)
('3_neutral 0.9-1.1', 719, 0.032, 0.241, 1.002, 53.3)
('4_high 1.1-1.3', 656, -0.005, 0.178, 1.115, 52.6)
('5_very_high >1.3', 724, 0.121, 0.332, 1.185, 60.1)

=== Max Pain Distance ===
('1_far_below <-3%', 36, -0.034, -1.55, 2.12, 63.9)
('2_below -3 to -1.5%', 115, 0.138, 0.71, 2.585, 57.4)
('3_slightly_below -1.5 to 0%', 819, 0.011, 0.212, 0.433, 49.6)
('4_slightly_above 0 to 1.5%', 1142, 0.069, 0.314, 1.179, 56.5)
('5_far_above >1.5%', 501, 0.041, 0.151, 1.101, 54.7)


In [44]:
# Futures - what tickers and how many rows
results = con.execute("""
    SELECT instrument_type, ticker, COUNT(*) as rows, 
           MIN(trade_date) as from_date, MAX(trade_date) as to_date
    FROM nse.futures_analytics
    GROUP BY instrument_type, ticker
    ORDER BY rows DESC
    LIMIT 10
""").fetchall()
print("=== Futures ===")
for row in results: print(row)

=== Futures ===
('STF', 'VOLTAS', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'GRASIM', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'TORNTPHARM', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'MARUTI', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'TITAN', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'SAIL', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'PAGEIND', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'AUROPHARMA', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'TATAPOWER', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))
('STF', 'HINDALCO', 7842, datetime.date(2016, 1, 1), datetime.date(2026, 7, 24))


In [45]:
# Participant - what participant types and asset classes exist
results = con.execute("""
    SELECT participant_type, metric_type, asset_class, direction, option_side,
           COUNT(*) as rows
    FROM nse.participant_activity
    GROUP BY participant_type, metric_type, asset_class, direction, option_side
    ORDER BY participant_type, metric_type, asset_class
    LIMIT 30
""").fetchall()
print("\n=== Participant ===")
for row in results: print(row)


=== Participant ===
('Client', 'OI', 'INDEX', 'long', 'NA', 2615)
('Client', 'OI', 'INDEX', 'short', 'NA', 2615)
('Client', 'OI', 'INDEX', 'short', 'PE', 2615)
('Client', 'OI', 'INDEX', 'short', 'CE', 2615)
('Client', 'OI', 'INDEX', 'long', 'PE', 2615)
('Client', 'OI', 'INDEX', 'long', 'CE', 2615)
('Client', 'OI', 'STOCK', 'long', 'CE', 2615)
('Client', 'OI', 'STOCK', 'short', 'CE', 2615)
('Client', 'OI', 'STOCK', 'long', 'NA', 2615)
('Client', 'OI', 'STOCK', 'long', 'PE', 2615)
('Client', 'OI', 'STOCK', 'short', 'PE', 2615)
('Client', 'OI', 'STOCK', 'short', 'NA', 2615)
('Client', 'VOL', 'INDEX', 'long', 'NA', 2615)
('Client', 'VOL', 'INDEX', 'short', 'PE', 2615)
('Client', 'VOL', 'INDEX', 'long', 'CE', 2615)
('Client', 'VOL', 'INDEX', 'long', 'PE', 2615)
('Client', 'VOL', 'INDEX', 'short', 'CE', 2615)
('Client', 'VOL', 'INDEX', 'short', 'NA', 2615)
('Client', 'VOL', 'STOCK', 'long', 'CE', 2615)
('Client', 'VOL', 'STOCK', 'short', 'PE', 2615)
('Client', 'VOL', 'STOCK', 'short', 'NA',

In [46]:
results = con.execute("""
  SELECT fa.trade_date, fa.expiry, fa.basis, fa.cost_of_carry, 
        fa.chng_oi_per, mdd.open_interest
  FROM nse.futures_analytics fa
  JOIN nse.instruments i
      ON i.ticker = fa.ticker
      AND i.instrument_type = fa.instrument_type
      AND i.expiry = fa.expiry
  JOIN nse.market_data_daily mdd
      ON mdd.instrument_key = i.instrument_key
      AND mdd.trade_date = fa.trade_date
  WHERE fa.ticker = 'NIFTY'
    AND fa.instrument_type = 'IDF'
    AND fa.trade_date = '2026-06-10'
  ORDER BY fa.expiry ASC
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 10), datetime.date(2026, 6, 30), 25.149999999997817, 0.019771203470175906, -1.4561936771066268, 19134310)
(datetime.date(2026, 6, 10), datetime.date(2026, 7, 28), 129.95000000000073, 0.042565737093267005, 0.20905923344947736, 1682460)
(datetime.date(2026, 6, 10), datetime.date(2026, 8, 25), 232.45000000000073, 0.04808848222918073, 1.3488880787458988, 542100)


In [47]:
con.execute("""
CREATE OR REPLACE TABLE participant_pivot AS
WITH base AS (
    SELECT
        trade_date,
        participant_type,
        metric_type,
        SUM(CASE WHEN asset_class='INDEX' AND option_side='NA' AND direction='long'  THEN contracts ELSE 0 END) AS idx_fut_long,
        SUM(CASE WHEN asset_class='INDEX' AND option_side='NA' AND direction='short' THEN contracts ELSE 0 END) AS idx_fut_short,
        SUM(CASE WHEN asset_class='STOCK' AND option_side='NA' AND direction='long'  THEN contracts ELSE 0 END) AS stk_fut_long,
        SUM(CASE WHEN asset_class='STOCK' AND option_side='NA' AND direction='short' THEN contracts ELSE 0 END) AS stk_fut_short,
        SUM(CASE WHEN asset_class='INDEX' AND option_side='CE' AND direction='long'  THEN contracts ELSE 0 END) AS idx_ce_long,
        SUM(CASE WHEN asset_class='INDEX' AND option_side='CE' AND direction='short' THEN contracts ELSE 0 END) AS idx_ce_short,
        SUM(CASE WHEN asset_class='INDEX' AND option_side='PE' AND direction='long'  THEN contracts ELSE 0 END) AS idx_pe_long,
        SUM(CASE WHEN asset_class='INDEX' AND option_side='PE' AND direction='short' THEN contracts ELSE 0 END) AS idx_pe_short
    FROM nse.participant_activity
    GROUP BY trade_date, participant_type, metric_type
),
derived AS (
    SELECT
        trade_date, participant_type, metric_type,
        ROUND(100.0 * (idx_fut_long - idx_fut_short) / NULLIF(idx_fut_long + idx_fut_short, 0), 2) AS idx_fut_net_pct,
        ROUND(100.0 * (stk_fut_long - stk_fut_short) / NULLIF(stk_fut_long + stk_fut_short, 0), 2) AS stk_fut_net_pct,
        ROUND(100.0 * ((idx_ce_long - idx_ce_short) - (idx_pe_long - idx_pe_short))
              / NULLIF((idx_ce_long + idx_ce_short + idx_pe_long + idx_pe_short), 0), 2) AS idx_opt_skew_pct
    FROM base
)
SELECT
    trade_date,

    MAX(CASE WHEN participant_type='FII'    AND metric_type='OI'  THEN idx_fut_net_pct END) AS fii_oi_idx_fut_net_pct,
    MAX(CASE WHEN participant_type='Client' AND metric_type='OI'  THEN idx_fut_net_pct END) AS client_oi_idx_fut_net_pct,
    MAX(CASE WHEN participant_type='DII'    AND metric_type='OI'  THEN idx_fut_net_pct END) AS dii_oi_idx_fut_net_pct,
    MAX(CASE WHEN participant_type='Pro'    AND metric_type='OI'  THEN idx_fut_net_pct END) AS pro_oi_idx_fut_net_pct,

    MAX(CASE WHEN participant_type='FII'    AND metric_type='OI'  THEN idx_opt_skew_pct END) AS fii_oi_idx_opt_skew_pct,
    MAX(CASE WHEN participant_type='Client' AND metric_type='OI'  THEN idx_opt_skew_pct END) AS client_oi_idx_opt_skew_pct,
    MAX(CASE WHEN participant_type='DII'    AND metric_type='OI'  THEN idx_opt_skew_pct END) AS dii_oi_idx_opt_skew_pct,
    MAX(CASE WHEN participant_type='Pro'    AND metric_type='OI'  THEN idx_opt_skew_pct END) AS pro_oi_idx_opt_skew_pct,

    MAX(CASE WHEN participant_type='FII'    AND metric_type='OI'  THEN stk_fut_net_pct END) AS fii_oi_stk_fut_net_pct,
    MAX(CASE WHEN participant_type='Client' AND metric_type='OI'  THEN stk_fut_net_pct END) AS client_oi_stk_fut_net_pct,

    -- VOL = daily flow/turnover, different signal than OI positioning
    MAX(CASE WHEN participant_type='FII'    AND metric_type='VOL' THEN idx_fut_net_pct END) AS fii_vol_idx_fut_net_pct,
    MAX(CASE WHEN participant_type='Client' AND metric_type='VOL' THEN idx_fut_net_pct END) AS client_vol_idx_fut_net_pct,
    MAX(CASE WHEN participant_type='Pro'    AND metric_type='VOL' THEN idx_fut_net_pct END) AS pro_vol_idx_fut_net_pct,
    MAX(CASE WHEN participant_type='FII'    AND metric_type='VOL' THEN idx_opt_skew_pct END) AS fii_vol_idx_opt_skew_pct,
    MAX(CASE WHEN participant_type='Client' AND metric_type='VOL' THEN idx_opt_skew_pct END) AS client_vol_idx_opt_skew_pct

FROM derived
GROUP BY trade_date
""")

results = con.execute("SELECT * FROM participant_pivot ORDER BY trade_date DESC LIMIT 5").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 24), -82.75, 58.24, 67.5, 29.32, -23.78, 5.78, -67.57, 1.8, 8.29, 83.97, -5.96, 5.64, 1.3, 0.44, -0.2)
(datetime.date(2026, 7, 23), -84.42, 56.14, 68.11, 30.6, -27.95, 7.95, -63.29, -1.18, 7.86, 81.96, -10.31, 2.02, 22.21, -0.77, 0.17)
(datetime.date(2026, 7, 22), -83.74, 56.5, 70.15, 17.43, -27.35, 8.18, -62.28, -1.17, 7.26, 81.27, -52.91, 15.61, 13.95, -1.71, 0.28)
(datetime.date(2026, 7, 21), -82.83, 52.55, 70.88, 6.82, -28.9, 9.04, -62.24, -1.74, 7.48, 81.76, -49.31, -1.94, -16.77, -0.37, 0.1)
(datetime.date(2026, 7, 20), -81.27, 53.11, 65.46, 14.44, -19.37, 4.02, -62.49, 1.66, 7.94, 82.04, -10.09, 2.99, 0.39, -0.58, 0.22)


In [49]:
con.execute("""
CREATE OR REPLACE TABLE daily_features AS
SELECT
    fr.trade_date,
    fr.close             AS nifty_close,
    fr.fwd_ret_1d,
    fr.fwd_ret_5d,
    fr.fwd_ret_20d,
    fr.up_1d,

    -- VIX
    vix.close            AS vix_close,

    -- PCR (all expiries)
    pcr_agg.pcr,

    -- Max pain distance (top-3 liquid expiries by rank, OI-weighted)
    mp_agg.max_pain_dist_pct,

    -- Futures near month
    fut.basis,
    fut.cost_of_carry,
    fut.chng_oi_per      AS fut_chng_oi_pct,
    
    fv.underlying_daily_vol,
    fv.futures_daily_vol,
    fv.applicable_daily_vol,

    -- Breadth
    mab.advances,
    mab.declines,
    ROUND(mab.advances::DOUBLE / NULLIF(mab.declines, 0), 4) AS adv_decl_ratio,
    mab.price_band_hits,

    -- Market summary
    mas.traded_value_cr,
    mas.num_trades,
    mas.market_cap_cr,

    -- FII net futures positioning (long - short, futures only)
    ROUND(
        (fii_long_fut.contracts - fii_short_fut.contracts) /
        NULLIF(fii_long_fut.contracts + fii_short_fut.contracts, 0) * 100
    , 2)                 AS fii_fut_net_pct,

    -- Client net futures positioning (retail, often contrarian)
    ROUND(
        (cli_long_fut.contracts - cli_short_fut.contracts) /
        NULLIF(cli_long_fut.contracts + cli_short_fut.contracts, 0) * 100
    , 2)                 AS client_fut_net_pct,

    -- FII put buying (protection signal)
    ROUND(
        (fii_long_pe.contracts - fii_short_pe.contracts) /
        NULLIF(fii_long_pe.contracts + fii_short_pe.contracts, 0) * 100
    , 2)                 AS fii_pe_net_pct,

    fii_flow.fii_net_flow_cr

FROM forward_returns fr

-- VIX
LEFT JOIN nse.market_activity_index vix
    ON vix.trade_date = fr.trade_date
    AND vix.index_name = 'INDIA VIX'

-- FO volatility
LEFT JOIN nse.fo_volatility fv
    ON fv.trade_date = fr.trade_date
    AND fv.ticker = 'NIFTY'

-- Market breadth
LEFT JOIN nse.market_activity_breadth mab
    ON mab.trade_date = fr.trade_date

-- Market activity summary
LEFT JOIN nse.market_activity_summary mas
    ON mas.trade_date = fr.trade_date

-- PCR: all expiries
LEFT JOIN (
    SELECT trade_date,
        ROUND(SUM(pe_oi) / NULLIF(SUM(ce_oi), 0), 4) AS pcr
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY' AND instrument_type = 'IDO'
    GROUP BY trade_date
) pcr_agg ON pcr_agg.trade_date = fr.trade_date

-- Max pain distance: top-3 most liquid expiries per date (relative, not absolute OI threshold)
LEFT JOIN (
    SELECT ranked.trade_date,
        ROUND(
            SUM(((ranked.underlying_price - ranked.max_pain) / NULLIF(ranked.max_pain, 0) * 100) * (ranked.pe_oi + ranked.ce_oi))
            / NULLIF(SUM(ranked.pe_oi + ranked.ce_oi), 0)
        , 4) AS max_pain_dist_pct
    FROM (
        SELECT oa.trade_date, oa.expiry, oa.max_pain, oa.pe_oi, oa.ce_oi, oa.underlying_price,
            ROW_NUMBER() OVER (PARTITION BY oa.trade_date ORDER BY (oa.pe_oi + oa.ce_oi) DESC) AS liquidity_rank
        FROM nse.options_analytics oa
        WHERE oa.ticker = 'NIFTY' AND oa.instrument_type = 'IDO'
    ) ranked
    WHERE ranked.liquidity_rank <= 3
    GROUP BY ranked.trade_date
) mp_agg ON mp_agg.trade_date = fr.trade_date

-- Futures: near month only
LEFT JOIN (
    SELECT fa.trade_date, fa.basis, fa.cost_of_carry, fa.chng_oi_per
    FROM nse.futures_analytics fa
    WHERE fa.ticker = 'NIFTY' AND fa.instrument_type = 'IDF'
    AND (fa.trade_date, fa.expiry) IN (
        SELECT trade_date, MIN(expiry)
        FROM nse.futures_analytics
        WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
        GROUP BY trade_date
    )
) fut ON fut.trade_date = fr.trade_date

-- Participant: FII long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) fii_long_fut ON fii_long_fut.trade_date = fr.trade_date

-- Participant: FII short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) fii_short_fut ON fii_short_fut.trade_date = fr.trade_date

-- Participant: Client long futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'NA'
) cli_long_fut ON cli_long_fut.trade_date = fr.trade_date

-- Participant: Client short futures
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'Client' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'NA'
) cli_short_fut ON cli_short_fut.trade_date = fr.trade_date

-- Participant: FII long PE (put buying = hedging/bearish)
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'long' AND option_side = 'PE'
) fii_long_pe ON fii_long_pe.trade_date = fr.trade_date

-- Participant: FII short PE
LEFT JOIN (
    SELECT trade_date, contracts FROM nse.participant_activity
    WHERE participant_type = 'FII' AND metric_type = 'OI'
      AND asset_class = 'INDEX' AND direction = 'short' AND option_side = 'PE'
) fii_short_pe ON fii_short_pe.trade_date = fr.trade_date

-- FII stats: net flow in Cr (index futures)
LEFT JOIN (
    SELECT trade_date, buy_amount_cr, sell_amount_cr,
           ROUND(buy_amount_cr - sell_amount_cr, 2) AS fii_net_flow_cr
    FROM nse.fii_stats
    WHERE instrument = 'INDEX FUTURES'
) fii_flow ON fii_flow.trade_date = fr.trade_date

ORDER BY fr.trade_date
""")

In [50]:
# 1. Add new columns without touching anything existing
con.execute("""
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS dii_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS pro_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_opt_skew_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS client_opt_skew_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS dii_opt_skew_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS pro_opt_skew_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_stk_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS client_stk_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_vol_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS client_vol_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS pro_vol_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_vol_opt_skew_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS client_vol_opt_skew_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_client_divergence DOUBLE;

    -- fii_stats table (separate source, kept distinctly named — not overwriting fii_fut_net_pct)
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_stats_fut_net_pct DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_stats_oi_net_cr DOUBLE;
""")

# 2. Fill from participant_pivot (built earlier)
con.execute("""
    UPDATE daily_features df
    SET dii_fut_net_pct         = pp.dii_oi_idx_fut_net_pct,
        pro_fut_net_pct         = pp.pro_oi_idx_fut_net_pct,
        fii_opt_skew_pct        = pp.fii_oi_idx_opt_skew_pct,
        client_opt_skew_pct     = pp.client_oi_idx_opt_skew_pct,
        dii_opt_skew_pct        = pp.dii_oi_idx_opt_skew_pct,
        pro_opt_skew_pct        = pp.pro_oi_idx_opt_skew_pct,
        fii_stk_fut_net_pct     = pp.fii_oi_stk_fut_net_pct,
        client_stk_fut_net_pct  = pp.client_oi_stk_fut_net_pct,
        fii_vol_fut_net_pct     = pp.fii_vol_idx_fut_net_pct,
        client_vol_fut_net_pct  = pp.client_vol_idx_fut_net_pct,
        pro_vol_fut_net_pct     = pp.pro_vol_idx_fut_net_pct,
        fii_vol_opt_skew_pct    = pp.fii_vol_idx_opt_skew_pct,
        client_vol_opt_skew_pct = pp.client_vol_idx_opt_skew_pct,
        fii_client_divergence   = ROUND(pp.fii_oi_idx_fut_net_pct - pp.client_oi_idx_fut_net_pct, 2)
    FROM participant_pivot pp
    WHERE df.trade_date = pp.trade_date
""")

# 3. Fill from fii_stats — INDEX FUTURES row, buy/sell contracts (distinct source from participant_activity)
con.execute("""
    UPDATE daily_features df
    SET fii_stats_fut_net_pct = ROUND(
            (fs.buy_contracts - fs.sell_contracts)::DOUBLE
            / NULLIF(fs.buy_contracts + fs.sell_contracts, 0) * 100
        , 2),
        fii_stats_oi_net_cr = fs.oi_amount_cr
    FROM (
        SELECT trade_date, buy_contracts, sell_contracts, oi_amount_cr
        FROM nse.fii_stats
        WHERE instrument = 'INDEX FUTURES'
    ) fs
    WHERE df.trade_date = fs.trade_date
""")

# sanity check — confirm nothing old was dropped and new cols populated
results = con.execute("""
    SELECT trade_date, fii_fut_net_pct, fii_pe_net_pct, fii_stats_fut_net_pct,
           dii_fut_net_pct, pro_fut_net_pct, fii_client_divergence
    FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 10
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 24), -82.75, 31.19, -5.96, 67.5, 29.32, -140.99)
(datetime.date(2026, 7, 23), -84.42, 35.75, -10.31, 68.11, 30.6, -140.56)
(datetime.date(2026, 7, 22), -83.74, 36.93, -52.91, 70.15, 17.43, -140.24)
(datetime.date(2026, 7, 21), -82.83, 41.23, -49.31, 70.88, 6.82, -135.38)
(datetime.date(2026, 7, 20), -81.27, 24.62, -10.09, 65.46, 14.44, -134.38)
(datetime.date(2026, 7, 17), -77.99, 23.42, 56.25, 65.47, 16.49, -127.94)
(datetime.date(2026, 7, 16), -83.09, 36.35, 39.53, 74.47, 22.67, -141.6)
(datetime.date(2026, 7, 15), -83.29, 38.9, 12.82, 75.07, 30.38, -143.15)
(datetime.date(2026, 7, 14), -83.27, 43.97, -40.25, 74.9, 27.45, -144.05)
(datetime.date(2026, 7, 13), -79.94, 26.78, -1.74, 75.58, 26.2, -137.29)


FILLING FO_VOLT for 2021-03-31

In [51]:
con.execute("""
    UPDATE daily_features
    SET underlying_daily_vol = (
            SELECT AVG(underlying_daily_vol) FROM daily_features
            WHERE trade_date IN ('2021-03-30', '2021-04-01')
        ),
        futures_daily_vol = (
            SELECT AVG(futures_daily_vol) FROM daily_features
            WHERE trade_date IN ('2021-03-30', '2021-04-01')
        ),
        applicable_daily_vol = (
            SELECT AVG(applicable_daily_vol) FROM daily_features
            WHERE trade_date IN ('2021-03-30', '2021-04-01')
        )
    WHERE trade_date = '2021-03-31'
      AND underlying_daily_vol IS NULL
""")

FILLING PARTICIPANT DATA for 2016-02-10

In [52]:
con.execute("""
    UPDATE daily_features
    SET fii_fut_net_pct = (
            SELECT AVG(fii_fut_net_pct) FROM daily_features
            WHERE trade_date IN ('2016-02-09', '2016-02-11')
        ),
        client_fut_net_pct = (
            SELECT AVG(client_fut_net_pct) FROM daily_features
            WHERE trade_date IN ('2016-02-09', '2016-02-11')
        ),
        fii_pe_net_pct = (
            SELECT AVG(fii_pe_net_pct) FROM daily_features
            WHERE trade_date IN ('2016-02-09', '2016-02-11')
        )
    WHERE trade_date = '2016-02-10'
      AND fii_fut_net_pct IS NULL
""")

In [56]:
con.execute("""
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS vix_chg_5d DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS pcr_chg_5d DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS basis_chg_5d DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS max_pain_dist_chg_5d DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS vix_realized_spread DOUBLE;
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS fii_fut_net_chg_5d DOUBLE;
""")

con.execute("""
    UPDATE daily_features df
    SET vix_chg_5d            = sub.vix_chg_5d,
        pcr_chg_5d             = sub.pcr_chg_5d,
        basis_chg_5d           = sub.basis_chg_5d,
        max_pain_dist_chg_5d   = sub.max_pain_dist_chg_5d,
        vix_realized_spread    = sub.vix_realized_spread,
        fii_fut_net_chg_5d     = sub.fii_fut_net_chg_5d
    FROM (
        SELECT
            trade_date,
            vix_close - LAG(vix_close, 5) OVER (ORDER BY trade_date) AS vix_chg_5d,
            pcr - LAG(pcr, 5) OVER (ORDER BY trade_date) AS pcr_chg_5d,
            basis - LAG(basis, 5) OVER (ORDER BY trade_date) AS basis_chg_5d,
            max_pain_dist_pct - LAG(max_pain_dist_pct, 5) OVER (ORDER BY trade_date) AS max_pain_dist_chg_5d,
            vix_close - (applicable_daily_vol * SQRT(252) * 100) AS vix_realized_spread,
            fii_fut_net_pct - LAG(fii_fut_net_pct, 5) OVER (ORDER BY trade_date) AS fii_fut_net_chg_5d
        FROM daily_features
    ) sub
    WHERE df.trade_date = sub.trade_date
""")

results = con.execute("""
    SELECT trade_date, vix_close, vix_chg_5d, vix_realized_spread, pcr_chg_5d
    FROM daily_features
    WHERE vix_chg_5d IS NOT NULL
    ORDER BY trade_date DESC LIMIT 5
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 7, 24), 14.03, 0.879999999999999, -0.04424342729199715, -0.5314)
(datetime.date(2026, 7, 23), 13.475, 0.5924999999999994, -0.6308971959775764, -0.1734)
(datetime.date(2026, 7, 22), 13.2925, 0.022500000000000853, -0.8386535379929967, -0.10370000000000001)
(datetime.date(2026, 7, 21), 12.6, -1.1500000000000004, -1.5385351841508683, -0.10550000000000004)
(datetime.date(2026, 7, 20), 12.98, -0.29999999999999893, -1.18950634899819, -0.15389999999999993)


In [57]:
# Full feature list — collinear vol cols dropped (#6): keep applicable_daily_vol only,
# drop underlying_daily_vol / futures_daily_vol (corr 0.999 with each other, 0.85 with applicable)
feature_cols = [
    # core market
    'vix_close', 'pcr', 'max_pain_dist_pct', 'basis', 'cost_of_carry', 'fut_chng_oi_pct',
    'applicable_daily_vol', 'adv_decl_ratio', 'traded_value_cr', 'num_trades', 'market_cap_cr',

    # FII / participant (from participant_activity)
    'fii_fut_net_pct', 'client_fut_net_pct', 'fii_pe_net_pct',
    'dii_fut_net_pct', 'pro_fut_net_pct',
    'fii_opt_skew_pct', 'client_opt_skew_pct', 'dii_opt_skew_pct', 'pro_opt_skew_pct',
    'fii_vol_fut_net_pct', 'client_vol_fut_net_pct', 'pro_vol_fut_net_pct',
    'fii_vol_opt_skew_pct', 'client_vol_opt_skew_pct',
    'fii_client_divergence',

    # fii_stats (separate source, kept distinct — low corr w/ fii_fut_net_pct, confirmed earlier)
    'fii_stats_fut_net_pct', 'fii_stats_oi_net_cr', 'fii_net_flow_cr',

    # momentum
    'vix_chg_5d', 'pcr_chg_5d', 'basis_chg_5d', 'max_pain_dist_chg_5d',
    'vix_realized_spread', 'fii_fut_net_chg_5d',
]

In [58]:
for col in feature_cols:
    n_nonnull = con.execute(f"""
        SELECT COUNT(*) FROM daily_features WHERE {col} IS NOT NULL
    """).fetchone()[0]
    print(f"{col:28s} {n_nonnull}")

vix_close                    2615
pcr                          2614
max_pain_dist_pct            2614
basis                        2614
cost_of_carry                2488
fut_chng_oi_pct              2614
applicable_daily_vol         2615
adv_decl_ratio               2615
traded_value_cr              2615
num_trades                   2615
market_cap_cr                2615
fii_fut_net_pct              2614
client_fut_net_pct           2615
fii_pe_net_pct               2614
dii_fut_net_pct              2614
pro_fut_net_pct              2614
fii_opt_skew_pct             2613
client_opt_skew_pct          2614
dii_opt_skew_pct             2614
pro_opt_skew_pct             2614
fii_vol_fut_net_pct          2614
client_vol_fut_net_pct       2614
pro_vol_fut_net_pct          2614
fii_vol_opt_skew_pct         2614
client_vol_opt_skew_pct      2614
fii_client_divergence        2613
fii_stats_fut_net_pct        2615
fii_stats_oi_net_cr          2615
fii_net_flow_cr              2615
vix_chg_5d    

In [59]:
# test ONLY the mp_agg join against forward_returns, isolated
test1 = con.execute("""
    SELECT COUNT(*) AS total_rows,
           COUNT(mp_agg.max_pain_dist_pct) AS non_null
    FROM forward_returns fr
    LEFT JOIN (
        SELECT ranked.trade_date,
            ROUND(
                SUM(((fr2.close - ranked.max_pain) / NULLIF(ranked.max_pain, 0) * 100) * (ranked.pe_oi + ranked.ce_oi))
                / NULLIF(SUM(ranked.pe_oi + ranked.ce_oi), 0)
            , 4) AS max_pain_dist_pct
        FROM (
            SELECT
                oa.trade_date, oa.expiry, oa.max_pain, oa.pe_oi, oa.ce_oi,
                ROW_NUMBER() OVER (
                    PARTITION BY oa.trade_date
                    ORDER BY (oa.pe_oi + oa.ce_oi) DESC
                ) AS liquidity_rank
            FROM nse.options_analytics oa
            WHERE oa.ticker = 'NIFTY' AND oa.instrument_type = 'IDO'
        ) ranked
        JOIN forward_returns fr2 ON fr2.trade_date = ranked.trade_date
        WHERE ranked.liquidity_rank <= 3
        GROUP BY ranked.trade_date
    ) mp_agg ON mp_agg.trade_date = fr.trade_date
""").fetchone()
print("mp_agg join test:", test1)

# test ONLY the futures near-month join against forward_returns, isolated
test2 = con.execute("""
    SELECT COUNT(*) AS total_rows,
           COUNT(fut.basis) AS non_null
    FROM forward_returns fr
    LEFT JOIN (
        SELECT fa.trade_date, fa.basis, fa.cost_of_carry, fa.chng_oi_per
        FROM nse.futures_analytics fa
        JOIN nse.instruments i
            ON i.ticker = fa.ticker AND i.instrument_type = fa.instrument_type AND i.expiry = fa.expiry
        JOIN nse.market_data_daily mdd
            ON mdd.instrument_key = i.instrument_key AND mdd.trade_date = fa.trade_date
        WHERE fa.ticker = 'NIFTY' AND fa.instrument_type = 'IDF'
          AND (fa.trade_date, fa.expiry) IN (
              SELECT trade_date, MIN(expiry)
              FROM nse.futures_analytics
              WHERE ticker = 'NIFTY' AND instrument_type = 'IDF'
              GROUP BY trade_date
          )
    ) fut ON fut.trade_date = fr.trade_date
""").fetchone()
print("fut join test:", test2)

mp_agg join test: (2615, 2614)
fut join test: (2615, 2614)


In [60]:
print("\nOPTIONS COVERAGE BY YEAR:")
for row in con.execute("""
    SELECT
        YEAR(trade_date) AS yr,
        COUNT(DISTINCT trade_date) AS dates
    FROM nse.options_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDO'
    GROUP BY YEAR(trade_date)
    ORDER BY yr
""").fetchall():
    print(row)

print("\nFUTURES COVERAGE BY YEAR:")
for row in con.execute("""
    SELECT
        YEAR(trade_date) AS yr,
        COUNT(DISTINCT trade_date) AS dates
    FROM nse.futures_analytics
    WHERE ticker = 'NIFTY'
      AND instrument_type = 'IDF'
    GROUP BY YEAR(trade_date)
    ORDER BY yr
""").fetchall():
    print(row)

print("\nFORWARD RETURNS BY YEAR:")
for row in con.execute("""
    SELECT
        YEAR(trade_date) AS yr,
        COUNT(DISTINCT trade_date) AS dates
    FROM forward_returns
    GROUP BY YEAR(trade_date)
    ORDER BY yr
""").fetchall():
    print(row)


OPTIONS COVERAGE BY YEAR:
(2016, 247)
(2017, 248)
(2018, 246)
(2019, 245)
(2020, 252)
(2021, 248)
(2022, 248)
(2023, 246)
(2024, 247)
(2025, 249)
(2026, 138)

FUTURES COVERAGE BY YEAR:
(2016, 247)
(2017, 248)
(2018, 246)
(2019, 245)
(2020, 252)
(2021, 248)
(2022, 248)
(2023, 246)
(2024, 247)
(2025, 249)
(2026, 138)

FORWARD RETURNS BY YEAR:
(2016, 247)
(2017, 248)
(2018, 246)
(2019, 245)
(2020, 252)
(2021, 248)
(2022, 248)
(2023, 246)
(2024, 248)
(2025, 249)
(2026, 138)


In [61]:
diag = con.execute("""
SELECT
    YEAR(trade_date) AS yr,

    COUNT(DISTINCT trade_date) AS dates,

    -- OPTIONS
    COUNT(DISTINCT CASE
        WHEN max_pain IS NOT NULL THEN trade_date
    END) AS opt_max_pain_dates,

    COUNT(DISTINCT CASE
        WHEN pe_oi IS NOT NULL THEN trade_date
    END) AS opt_pe_oi_dates,

    COUNT(DISTINCT CASE
        WHEN ce_oi IS NOT NULL THEN trade_date
    END) AS opt_ce_oi_dates

FROM nse.options_analytics

WHERE ticker = 'NIFTY'
  AND instrument_type = 'IDO'

GROUP BY YEAR(trade_date)
ORDER BY yr
""").fetchall()

print("OPTIONS")
print("year | dates | max_pain | pe_oi | ce_oi")
for r in diag:
    print(r)


diag = con.execute("""
SELECT
    YEAR(trade_date) AS yr,

    COUNT(DISTINCT trade_date) AS dates,

    COUNT(DISTINCT CASE
        WHEN basis IS NOT NULL THEN trade_date
    END) AS basis_dates,

    COUNT(DISTINCT CASE
        WHEN cost_of_carry IS NOT NULL THEN trade_date
    END) AS coc_dates,

    COUNT(DISTINCT CASE
        WHEN chng_oi_per IS NOT NULL THEN trade_date
    END) AS oi_change_dates

FROM nse.futures_analytics

WHERE ticker = 'NIFTY'
  AND instrument_type = 'IDF'

GROUP BY YEAR(trade_date)
ORDER BY yr
""").fetchall()

print("\nFUTURES")
print("year | dates | basis | coc | chng_oi")
for r in diag:
    print(r)

OPTIONS
year | dates | max_pain | pe_oi | ce_oi
(2016, 247, 247, 247, 247)
(2017, 248, 248, 248, 248)
(2018, 246, 246, 246, 246)
(2019, 245, 245, 245, 245)
(2020, 252, 252, 252, 252)
(2021, 248, 248, 248, 248)
(2022, 248, 248, 248, 248)
(2023, 246, 246, 246, 246)
(2024, 247, 247, 247, 247)
(2025, 249, 249, 249, 249)
(2026, 138, 138, 138, 138)

FUTURES
year | dates | basis | coc | chng_oi
(2016, 247, 247, 247, 247)
(2017, 248, 248, 248, 248)
(2018, 246, 246, 246, 246)
(2019, 245, 245, 245, 245)
(2020, 252, 252, 252, 252)
(2021, 248, 248, 248, 248)
(2022, 248, 248, 248, 248)
(2023, 246, 246, 246, 246)
(2024, 247, 247, 247, 247)
(2025, 249, 249, 249, 249)
(2026, 138, 138, 138, 138)


In [62]:
con.execute("""
SELECT
    MIN(trade_date) FILTER (WHERE max_pain IS NOT NULL) AS first_max_pain,
    MAX(trade_date) FILTER (WHERE max_pain IS NULL) AS last_missing_max_pain
FROM nse.options_analytics
WHERE ticker = 'NIFTY';

SELECT
    MIN(trade_date) FILTER (WHERE basis IS NOT NULL) AS first_basis,
    MIN(trade_date) FILTER (WHERE cost_of_carry IS NOT NULL) AS first_coc,
    MAX(trade_date) FILTER (WHERE basis IS NULL) AS last_missing_basis
FROM nse.futures_analytics
WHERE ticker = 'NIFTY';
""").fetchall()

[(datetime.date(2016, 1, 1), datetime.date(2016, 1, 1), None)]

In [63]:
df_corr = con.execute("""
    SELECT fii_fut_net_pct, fii_stats_fut_net_pct
    FROM daily_features
    WHERE fii_fut_net_pct IS NOT NULL AND fii_stats_fut_net_pct IS NOT NULL
""").df()

print(df_corr.corr())
print("n =", len(df_corr))

                       fii_fut_net_pct  fii_stats_fut_net_pct
fii_fut_net_pct               1.000000               0.078815
fii_stats_fut_net_pct         0.078815               1.000000
n = 2614


In [64]:
import scipy.stats as stats

for col in ['fii_fut_net_pct', 'fii_stats_fut_net_pct']:
    sub = con.execute(f"""
        SELECT {col}, fwd_ret_5d FROM daily_features
        WHERE {col} IS NOT NULL AND fwd_ret_5d IS NOT NULL
    """).df()
    ic, p = stats.spearmanr(sub[col], sub['fwd_ret_5d'])
    print(f"{col:25s} IC={ic:.4f} p={p:.4f} n={len(sub)}")

fii_fut_net_pct           IC=0.0267 p=0.1730 n=2609
fii_stats_fut_net_pct     IC=0.0182 p=0.3516 n=2610


In [65]:
def run_and_flag(query, min_n=30):
    rows = con.execute(query).fetchall()
    for r in rows:
        n = r[1]  # assumes 2nd column is `days` count, adjust index if needed
        flag = "  ⚠ LOW N" if n < min_n else ""
        print(r, flag)

In [66]:
# Check if participant_activity trade_dates align with trading calendar cleanly
# vs any evidence of same-day-availability (compare against vix or nifty close dates)
results = con.execute("""
    SELECT pa.trade_date, COUNT(*) 
    FROM nse.participant_activity pa
    LEFT JOIN forward_returns fr ON fr.trade_date = pa.trade_date
    WHERE fr.trade_date IS NULL
    GROUP BY pa.trade_date
    ORDER BY pa.trade_date
    LIMIT 20
""").fetchall()
for row in results: print(row)

(datetime.date(2024, 2, 5), 96)


In [67]:
results = con.execute("""
    SELECT trade_date, underlying_daily_vol, futures_daily_vol, applicable_daily_vol
    FROM daily_features
    WHERE trade_date BETWEEN '2016-02-08' AND '2016-02-12'
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2016, 2, 8), 0.0107, 0.0107, 0.0107)
(datetime.date(2016, 2, 9), 0.0108, 0.0107, 0.0108)
(datetime.date(2016, 2, 10), 0.0108, 0.0107, 0.0108)
(datetime.date(2016, 2, 11), 0.0133, 0.0135, 0.0135)
(datetime.date(2016, 2, 12), 0.0129, 0.0131, 0.0131)


In [68]:
results = con.execute("""
    SELECT trade_date, fii_fut_net_pct, client_fut_net_pct, fii_pe_net_pct
    FROM daily_features
    WHERE trade_date BETWEEN '2016-02-08' AND '2016-02-12'
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2016, 2, 8), -17.07, -0.07, 66.11)
(datetime.date(2016, 2, 9), -12.57, -1.54, 70.85)
(datetime.date(2016, 2, 10), -12.105, 2.33, 72.715)
(datetime.date(2016, 2, 11), -11.64, 6.2, 74.58)
(datetime.date(2016, 2, 12), -4.81, 4.1, 73.11)


In [70]:
results = con.execute("""
    SELECT * FROM daily_features
    WHERE vix_close IS NOT NULL
    ORDER BY trade_date DESC LIMIT 5
    OFFSET 20;
""").fetchall()
for row in results: print(row)

(datetime.date(2026, 6, 25), 24056.0, -0.4562, 0.8931, -1.1995, 0, 13.05, 1.0608, -0.1826, 46.29999999999927, 0.14050133022946237, -24.29122148926378, 0.00894584, 0.00903173, 0.00903173, 1281, 2174, 0.5892, 197, 136051.69, 37874452, 47479950.52, -68.6, 48.3, 33.66, 810.74, 73.36, 3.22, -25.61, 7.56, -50.25, 0.01, 8.72, 82.94, 3.06, -1.71, -6.08, 0.34, 0.36, -116.9, 3.06, 52753.71, 0.3775000000000013, -0.062100000000000044, 21.799999999999272, -0.5690000000000001, -1.2874268932088366, 4.450000000000003)
(datetime.date(2026, 6, 24), 24021.65, 0.143, 0.6413, -0.633, 1, 13.385, 1.2085, -0.1447, 30.149999999997817, 0.0763529982328386, -17.269656711223927, 0.00896764, 0.00905296, 0.00905296, 1861, 1675, 1.111, 213, 122776.22, 37727066, 47606684.71, -70.96, 48.67, 34.38, -45.53, 72.63, 8.14, -28.34, 6.63, 13.1, 5.19, 8.46, 83.36, -0.37, -4.72, 13.68, 0.6, -0.71, -119.63, -0.37, 51992.19, 0.1974999999999998, 0.10849999999999982, 21.849999999998545, -0.5018, -0.9861284734091793, 2.8599999999999

In [71]:
queries = {
    "Basis": ("basis", [
        ("1_discount <0",      "basis < 0"),
        ("2_low 0-50",         "basis >= 0 AND basis < 50"),
        ("3_mid 50-100",       "basis >= 50 AND basis < 100"),
        ("4_high >100",        "basis >= 100"),
    ]),
    "FII Fut Net": ("fii_fut_net_pct", [
        ("1_very_short <-50",  "fii_fut_net_pct < -50"),
        ("2_short -50 to -20", "fii_fut_net_pct >= -50 AND fii_fut_net_pct < -20"),
        ("3_neutral -20 to 20","fii_fut_net_pct >= -20 AND fii_fut_net_pct < 20"),
        ("4_long 20 to 50",    "fii_fut_net_pct >= 20 AND fii_fut_net_pct < 50"),
        ("5_very_long >50",    "fii_fut_net_pct >= 50"),
    ]),
    "Client Fut Net": ("client_fut_net_pct", [
        ("1_very_short <-50",  "client_fut_net_pct < -50"),
        ("2_short -50 to -20", "client_fut_net_pct >= -50 AND client_fut_net_pct < -20"),
        ("3_neutral -20 to 20","client_fut_net_pct >= -20 AND client_fut_net_pct < 20"),
        ("4_long 20 to 50",    "client_fut_net_pct >= 20 AND client_fut_net_pct < 50"),
        ("5_very_long >50",    "client_fut_net_pct >= 50"),
    ]),
    "FII PE Net": ("fii_pe_net_pct", [
        ("1_very_short <-50",  "fii_pe_net_pct < -50"),
        ("2_short -50 to -20", "fii_pe_net_pct >= -50 AND fii_pe_net_pct < -20"),
        ("3_neutral -20 to 20","fii_pe_net_pct >= -20 AND fii_pe_net_pct < 20"),
        ("4_long 20 to 50",    "fii_pe_net_pct >= 20 AND fii_pe_net_pct < 50"),
        ("5_very_long >50",    "fii_pe_net_pct >= 50"),
    ]),
}

for factor_name, (col, buckets) in queries.items():
    case_sql = "CASE\n" + "\n".join(
        f"  WHEN {cond} THEN '{label}'" for label, cond in buckets
    ) + "\n  ELSE 'other' END"

    results = con.execute(f"""
        SELECT
            {case_sql} AS bucket,
            COUNT(*)                    AS days,
            ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
            ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
            ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
            ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
        FROM daily_features
        WHERE fwd_ret_1d IS NOT NULL AND {col} IS NOT NULL
        GROUP BY 1 ORDER BY 1
    """).fetchall()

    print(f"\n=== {factor_name} ===")
    for row in results: print(row)


=== Basis ===
('1_discount <0', 390, 0.03, 0.26, 1.15, 55.6)
('2_low 0-50', 1621, 0.064, 0.31, 1.176, 54.3)
('3_mid 50-100', 457, 0.012, 0.094, 0.58, 52.1)
('4_high >100', 145, 0.019, -0.094, 0.107, 53.8)

=== FII Fut Net ===
('1_very_short <-50', 522, -0.008, 0.014, 0.092, 50.2)
('2_short -50 to -20', 347, 0.051, 0.176, 1.294, 53.9)
('3_neutral -20 to 20', 946, 0.061, 0.362, 1.225, 54.4)
('4_long 20 to 50', 564, 0.074, 0.263, 1.308, 57.6)
('5_very_long >50', 234, 0.039, 0.298, 0.955, 53.4)

=== Client Fut Net ===
('1_very_short <-50', 29, 0.041, 0.172, -0.381, 55.2)
('2_short -50 to -20', 243, 0.091, 0.237, 1.137, 54.3)
('3_neutral -20 to 20', 1700, 0.068, 0.377, 1.458, 55.2)
('4_long 20 to 50', 595, -0.031, -0.179, -0.326, 50.8)
('5_very_long >50', 47, 0.057, 0.809, 2.299, 57.4)

=== FII PE Net ===
('3_neutral -20 to 20', 596, 0.13, 0.382, 1.16, 60.1)
('4_long 20 to 50', 1699, 0.011, 0.176, 0.909, 52.1)
('5_very_long >50', 318, 0.08, 0.325, 1.248, 53.8)


In [72]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    ORDER BY 1, 2
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_discount', 168, -0.129, 0.131, 2.088, 50.0)
('VIX_high', 'basis_high', 10, 0.059, 1.27, 2.324, 70.0)
('VIX_high', 'basis_low', 455, 0.148, 0.582, 2.497, 56.0)
('VIX_high', 'basis_mid', 73, 0.147, 0.549, 1.163, 54.8)
('VIX_low', 'basis_discount', 123, 0.076, 0.336, 0.511, 55.3)
('VIX_low', 'basis_high', 96, 0.008, -0.206, 0.054, 52.1)
('VIX_low', 'basis_low', 525, 0.023, 0.196, 0.766, 53.7)
('VIX_low', 'basis_mid', 237, 0.012, 0.087, 0.621, 56.1)
('VIX_mid', 'basis_discount', 99, 0.243, 0.387, 0.314, 65.7)
('VIX_mid', 'basis_high', 39, 0.036, -0.17, -0.33, 53.8)
('VIX_mid', 'basis_low', 641, 0.038, 0.211, 0.567, 53.7)
('VIX_mid', 'basis_mid', 147, -0.056, -0.12, 0.225, 44.2)


In [73]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        CASE
            WHEN fii_fut_net_pct < -50 THEN 'FII_short'
            WHEN fii_fut_net_pct < 20  THEN 'FII_neutral'
            ELSE                            'FII_long'
        END AS fii_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL
      AND fii_fut_net_pct IS NOT NULL

    GROUP BY 1, 2, 3
    HAVING COUNT(*) >= 8      -- filter cells too small to mean anything
    ORDER BY avg_ret_20d DESC
""").fetchall()

for row in results: print(row)

('VIX_high', 'basis_discount', 'FII_neutral', 101, -0.036, 0.772, 3.163, 46.5)
('VIX_high', 'basis_discount', 'FII_long', 28, 0.115, 0.729, 2.949, 67.9)
('VIX_high', 'basis_low', 'FII_long', 130, 0.076, 0.479, 2.796, 61.5)
('VIX_high', 'basis_low', 'FII_short', 55, 0.156, 0.31, 2.448, 54.5)
('VIX_high', 'basis_low', 'FII_neutral', 270, 0.181, 0.686, 2.364, 53.7)
('VIX_high', 'basis_mid', 'FII_neutral', 29, 0.289, 0.655, 2.248, 65.5)
('VIX_high', 'basis_mid', 'FII_short', 14, -0.214, 1.007, 2.188, 42.9)
('VIX_low', 'basis_high', 'FII_neutral', 30, 0.127, -0.281, 1.55, 56.7)
('VIX_low', 'basis_discount', 'FII_long', 40, 0.126, 0.713, 1.538, 62.5)
('VIX_mid', 'basis_discount', 'FII_neutral', 59, 0.327, 0.501, 1.483, 69.5)
('VIX_low', 'basis_mid', 'FII_short', 101, 0.04, 0.323, 1.377, 54.5)
('VIX_low', 'basis_low', 'FII_long', 213, 0.047, 0.308, 1.294, 52.6)
('VIX_low', 'basis_discount', 'FII_neutral', 65, 0.054, 0.151, 1.19, 53.8)
('VIX_mid', 'basis_discount', 'FII_long', 25, -0.012, 0.40

In [74]:
import scipy.stats as stats
import pandas as pd

df = con.execute("""
    SELECT vix_close, pcr, max_pain_dist_pct, basis, 
           cost_of_carry, fut_chng_oi_pct,
           fii_fut_net_pct, client_fut_net_pct,
           underlying_daily_vol, futures_daily_vol, applicable_daily_vol,
           adv_decl_ratio, traded_value_cr, num_trades, market_cap_cr, fii_net_flow_cr,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
""").df()

factors = ['vix_close', 'pcr', 'max_pain_dist_pct', 'basis',
           'cost_of_carry', 'fut_chng_oi_pct', 'fii_fut_net_pct', 
           'client_fut_net_pct', 'underlying_daily_vol', 
           'futures_daily_vol', 'applicable_daily_vol',
           'adv_decl_ratio', 'traded_value_cr', 'num_trades', 'market_cap_cr', 'fii_net_flow_cr']

targets = ['fwd_ret_1d', 'fwd_ret_5d', 'fwd_ret_20d']

rows = []
for f in factors:
    row = {'factor': f}
    for t in targets:
        mask = df[f].notna() & df[t].notna()
        ic, pval = stats.spearmanr(df.loc[mask, f], df.loc[mask, t])
        row[f'IC_{t}'] = round(ic, 4)
        row[f'pval_{t}'] = round(pval, 4)
    rows.append(row)

ic_df = pd.DataFrame(rows)
print(ic_df.to_string(index=False))

              factor  IC_fwd_ret_1d  pval_fwd_ret_1d  IC_fwd_ret_5d  pval_fwd_ret_5d  IC_fwd_ret_20d  pval_fwd_ret_20d
           vix_close         0.0509           0.0093         0.0891           0.0000          0.1661            0.0000
                 pcr         0.0449           0.0216         0.0245           0.2116          0.0620            0.0016
   max_pain_dist_pct         0.0052           0.7913        -0.0327           0.0945         -0.0315            0.1088
               basis        -0.0414           0.0345        -0.0923           0.0000         -0.1396            0.0000
       cost_of_carry        -0.0355           0.0768        -0.0423           0.0349         -0.0901            0.0000
     fut_chng_oi_pct        -0.0245           0.2100        -0.0460           0.0188         -0.0018            0.9270
     fii_fut_net_pct         0.0293           0.1339         0.0267           0.1730          0.0359            0.0674
  client_fut_net_pct        -0.0323           0.

In [75]:
print(df[['vix_close','underlying_daily_vol','futures_daily_vol','applicable_daily_vol']].corr())

                      vix_close  underlying_daily_vol  futures_daily_vol  \
vix_close              1.000000              0.848149           0.850052   
underlying_daily_vol   0.848149              1.000000           0.999348   
futures_daily_vol      0.850052              0.999348           1.000000   
applicable_daily_vol   0.850603              0.999583           0.999873   

                      applicable_daily_vol  
vix_close                         0.850603  
underlying_daily_vol              0.999583  
futures_daily_vol                 0.999873  
applicable_daily_vol              1.000000  


In [76]:
# IC of max pain dist conditioned on being away from max pain
mask_tail = df['max_pain_dist_pct'].abs() > 1.5
sub = df[mask_tail & df['fwd_ret_20d'].notna() & df['max_pain_dist_pct'].notna()]
ic, pval = stats.spearmanr(sub['max_pain_dist_pct'], sub['fwd_ret_20d'])
print(f"Max pain tail IC (20D): {ic:.4f}, p={pval:.4f}, n={len(sub)}")

Max pain tail IC (20D): -0.2811, p=0.0000, n=716


In [77]:
TRAIN_END = '2025-06-20'
TEST_START = '2025-06-21'

# Verify the split
results = con.execute(f"""
    SELECT 
        CASE WHEN trade_date <= '{TRAIN_END}' THEN 'train' ELSE 'test' END AS split,
        COUNT(*) as days,
        MIN(trade_date) as from_date,
        MAX(trade_date) as to_date
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
    GROUP BY 1
    ORDER BY 1
""").fetchall()
for row in results: print(row)

('test', 269, datetime.date(2025, 6, 23), datetime.date(2026, 7, 23))
('train', 2345, datetime.date(2016, 1, 1), datetime.date(2025, 6, 20))


In [78]:
con.execute(f"""
    ALTER TABLE daily_features ADD COLUMN IF NOT EXISTS split VARCHAR;
    
    UPDATE daily_features 
    SET split = CASE 
        WHEN trade_date <= '{TRAIN_END}' THEN 'train' 
        ELSE 'test' 
    END
""")

In [79]:
df_train = con.execute("""
    SELECT vix_close, pcr, max_pain_dist_pct, basis, 
           cost_of_carry, fut_chng_oi_pct,
           fii_fut_net_pct, client_fut_net_pct,
           underlying_daily_vol, futures_daily_vol, applicable_daily_vol,
           adv_decl_ratio, traded_value_cr, num_trades, market_cap_cr, fii_net_flow_cr,
           fwd_ret_1d, fwd_ret_5d, fwd_ret_20d
    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND split = 'train'
""").df()

rows = []
for f in factors:
    row = {'factor': f}
    for t in targets:
        mask = df_train[f].notna() & df_train[t].notna()
        ic, pval = stats.spearmanr(df_train.loc[mask, f], df_train.loc[mask, t])
        row[f'IC_{t}'] = round(ic, 4)
        row[f'pval_{t}'] = round(pval, 4)
        row[f'n_{t}'] = mask.sum()
    rows.append(row)

ic_df_train = pd.DataFrame(rows)
print("=== IC Analysis — TRAIN ONLY (Jun 2024 - Jun 2025) ===")
print(ic_df_train.to_string(index=False))

=== IC Analysis — TRAIN ONLY (Jun 2024 - Jun 2025) ===
              factor  IC_fwd_ret_1d  pval_fwd_ret_1d  n_fwd_ret_1d  IC_fwd_ret_5d  pval_fwd_ret_5d  n_fwd_ret_5d  IC_fwd_ret_20d  pval_fwd_ret_20d  n_fwd_ret_20d
           vix_close         0.0446           0.0308          2345         0.0845           0.0000          2345          0.1473            0.0000           2345
                 pcr         0.0425           0.0398          2344         0.0084           0.6860          2344          0.0411            0.0464           2344
   max_pain_dist_pct        -0.0015           0.9414          2344        -0.0599           0.0037          2344         -0.0609            0.0032           2344
               basis        -0.0343           0.0965          2344        -0.0862           0.0000          2344         -0.1236            0.0000           2344
       cost_of_carry        -0.0361           0.0887          2231        -0.0427           0.0436          2231         -0.0912       

In [80]:
# total row count vs distinct dates — catches fanout duplication
total_rows = con.execute("SELECT COUNT(*) FROM daily_features").fetchone()
distinct_dates = con.execute("SELECT COUNT(DISTINCT trade_date) FROM daily_features").fetchone()
print("total rows:", total_rows, "distinct dates:", distinct_dates)

# date range where basis/max_pain actually ARE populated
range_check = con.execute("""
    SELECT
        MIN(trade_date) FILTER (WHERE basis IS NOT NULL) AS basis_min,
        MAX(trade_date) FILTER (WHERE basis IS NOT NULL) AS basis_max,
        MIN(trade_date) FILTER (WHERE max_pain_dist_pct IS NOT NULL) AS mp_min,
        MAX(trade_date) FILTER (WHERE max_pain_dist_pct IS NOT NULL) AS mp_max
    FROM daily_features
""").fetchone()
print("basis populated range:", range_check[0], "to", range_check[1])
print("max_pain populated range:", range_check[2], "to", range_check[3])

total rows: (2615,) distinct dates: (2615,)
basis populated range: 2016-01-01 to 2026-07-24
max_pain populated range: 2016-01-01 to 2026-07-24


In [81]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
import scipy.stats as stats

target = 'fwd_ret_5d'

df_all = con.execute(f"""
    SELECT {','.join(feature_cols)}, {target}, split
    FROM daily_features
    WHERE {target} IS NOT NULL
""").df()

train = df_all[df_all.split == 'train'].dropna()
test = df_all[df_all.split == 'test'].dropna()

print(f"train n={len(train)}, test n={len(test)}")

model = GradientBoostingRegressor(
    max_depth=3, n_estimators=200, learning_rate=0.03,
    subsample=0.8, random_state=0
)
model.fit(train[feature_cols], train[target])

train_pred = model.predict(train[feature_cols])
test_pred = model.predict(test[feature_cols])

train_ic, _ = stats.spearmanr(train_pred, train[target])
test_ic, _ = stats.spearmanr(test_pred, test[target])
print(f"\nModel IC — train: {train_ic:.4f}, test: {test_ic:.4f}")
print("(if test IC is much lower than train, model is overfitting)")

imp = permutation_importance(
    model, test[feature_cols], test[target],
    n_repeats=30, random_state=0
)

print(f"\n=== Permutation importance on TEST (target={target}) ===")
order = imp.importances_mean.argsort()[::-1]
for i in order:
    mean, std = imp.importances_mean[i], imp.importances_std[i]
    flag = "" if mean - std > 0 else "  ⚠ not distinguishable from 0"
    print(f"{feature_cols[i]:28s} {mean:8.5f} ± {std:.5f}{flag}")

train n=2222, test n=252

Model IC — train: 0.6146, test: -0.0878
(if test IC is much lower than train, model is overfitting)

=== Permutation importance on TEST (target=fwd_ret_5d) ===
vix_chg_5d                    0.18201 ± 0.05825
fii_stats_oi_net_cr           0.11124 ± 0.01614
pro_opt_skew_pct              0.02449 ± 0.01995
adv_decl_ratio                0.01548 ± 0.00356
num_trades                    0.00756 ± 0.00642
pro_vol_fut_net_pct           0.00745 ± 0.01525  ⚠ not distinguishable from 0
vix_realized_spread           0.00736 ± 0.00700
traded_value_cr               0.00685 ± 0.00237
market_cap_cr                 0.00554 ± 0.01577  ⚠ not distinguishable from 0
fii_fut_net_chg_5d            0.00530 ± 0.00540  ⚠ not distinguishable from 0
cost_of_carry                 0.00469 ± 0.00426
basis_chg_5d                  0.00374 ± 0.00434  ⚠ not distinguishable from 0
max_pain_dist_chg_5d          0.00369 ± 0.00378  ⚠ not distinguishable from 0
fii_fut_net_pct               0.00166 ± 

In [82]:
target = 'fwd_ret_1d'

df_all = con.execute(f"""
    SELECT {','.join(feature_cols)}, {target}, split
    FROM daily_features
    WHERE {target} IS NOT NULL
""").df()

train = df_all[df_all.split == 'train'].dropna()
test = df_all[df_all.split == 'test'].dropna()

print(f"train n={len(train)}, test n={len(test)}")

model = GradientBoostingRegressor(
    max_depth=3, n_estimators=200, learning_rate=0.03,
    subsample=0.8, random_state=0
)
model.fit(train[feature_cols], train[target])

train_pred = model.predict(train[feature_cols])
test_pred = model.predict(test[feature_cols])

train_ic, _ = stats.spearmanr(train_pred, train[target])
test_ic, _ = stats.spearmanr(test_pred, test[target])
print(f"\nModel IC — train: {train_ic:.4f}, test: {test_ic:.4f}")
print("(if test IC is much lower than train, model is overfitting)")

imp = permutation_importance(
    model, test[feature_cols], test[target],
    n_repeats=30, random_state=0
)

print(f"\n=== Permutation importance on TEST (target={target}) ===")
order = imp.importances_mean.argsort()[::-1]
for i in order:
    mean, std = imp.importances_mean[i], imp.importances_std[i]
    flag = "" if mean - std > 0 else "  ⚠ not distinguishable from 0"
    print(f"{feature_cols[i]:28s} {mean:8.5f} ± {std:.5f}{flag}")

train n=2222, test n=256

Model IC — train: 0.3715, test: -0.1103
(if test IC is much lower than train, model is overfitting)

=== Permutation importance on TEST (target=fwd_ret_1d) ===
client_vol_fut_net_pct        0.03578 ± 0.03980  ⚠ not distinguishable from 0
traded_value_cr               0.01995 ± 0.01188
vix_close                     0.01664 ± 0.00942
vix_chg_5d                    0.01526 ± 0.01542  ⚠ not distinguishable from 0
fii_vol_opt_skew_pct          0.01388 ± 0.03174  ⚠ not distinguishable from 0
fii_pe_net_pct                0.01103 ± 0.00345
fii_fut_net_pct               0.01101 ± 0.00535
max_pain_dist_pct             0.00931 ± 0.00521
fii_net_flow_cr               0.00744 ± 0.00611
pro_opt_skew_pct              0.00694 ± 0.00797  ⚠ not distinguishable from 0
pro_vol_fut_net_pct           0.00537 ± 0.00714  ⚠ not distinguishable from 0
cost_of_carry                 0.00315 ± 0.00198
pro_fut_net_pct               0.00247 ± 0.00616  ⚠ not distinguishable from 0
basis_chg_

In [83]:
# Full feature list — collinear vol cols dropped (#6): keep applicable_daily_vol only,
# drop underlying_daily_vol / futures_daily_vol (corr 0.999 with each other, 0.85 with applicable)
feature_cols = [
    # core market
    'vix_close', 'pcr', 'max_pain_dist_pct', 'basis', 'cost_of_carry', 'fut_chng_oi_pct',
    'applicable_daily_vol', 'adv_decl_ratio', 'traded_value_cr', 'num_trades', 'market_cap_cr',

    # FII / participant (from participant_activity)
    'fii_fut_net_pct', 'client_fut_net_pct', 'fii_pe_net_pct',
    'dii_fut_net_pct', 'pro_fut_net_pct',
    'fii_opt_skew_pct', 'client_opt_skew_pct', 'dii_opt_skew_pct', 'pro_opt_skew_pct',
    'fii_vol_fut_net_pct', 'client_vol_fut_net_pct', 'pro_vol_fut_net_pct',
    'fii_vol_opt_skew_pct', 'client_vol_opt_skew_pct',
    'fii_client_divergence',

    # fii_stats (separate source, kept distinct — low corr w/ fii_fut_net_pct, confirmed earlier)
    'fii_stats_fut_net_pct', 'fii_stats_oi_net_cr', 'fii_net_flow_cr',

    # momentum
    'vix_chg_5d', 'pcr_chg_5d', 'basis_chg_5d', 'max_pain_dist_chg_5d',
    'vix_realized_spread', 'fii_fut_net_chg_5d',
]

target = 'fwd_ret_20d'

df_all = con.execute(f"""
    SELECT {','.join(feature_cols)}, {target}, split
    FROM daily_features
    WHERE {target} IS NOT NULL
""").df()

train = df_all[df_all.split == 'train'].dropna()
test = df_all[df_all.split == 'test'].dropna()

print(f"train n={len(train)}, test n={len(test)}")

model = GradientBoostingRegressor(
    max_depth=3, n_estimators=200, learning_rate=0.03,
    subsample=0.8, random_state=0
)
model.fit(train[feature_cols], train[target])

train_pred = model.predict(train[feature_cols])
test_pred = model.predict(test[feature_cols])

train_ic, _ = stats.spearmanr(train_pred, train[target])
test_ic, _ = stats.spearmanr(test_pred, test[target])
print(f"\nModel IC — train: {train_ic:.4f}, test: {test_ic:.4f}")
print("(if test IC is much lower than train, model is overfitting)")

imp = permutation_importance(
    model, test[feature_cols], test[target],
    n_repeats=30, random_state=0
)

print(f"\n=== Permutation importance on TEST (target={target}) ===")
order = imp.importances_mean.argsort()[::-1]
for i in order:
    mean, std = imp.importances_mean[i], imp.importances_std[i]
    flag = "" if mean - std > 0 else "  ⚠ not distinguishable from 0"
    print(f"{feature_cols[i]:28s} {mean:8.5f} ± {std:.5f}{flag}")

train n=2222, test n=238

Model IC — train: 0.7787, test: -0.4258
(if test IC is much lower than train, model is overfitting)

=== Permutation importance on TEST (target=fwd_ret_20d) ===
market_cap_cr                 0.16155 ± 0.03280
fii_fut_net_pct               0.04257 ± 0.00819
fii_stats_oi_net_cr           0.03663 ± 0.00714
max_pain_dist_pct             0.03588 ± 0.01391
pro_fut_net_pct               0.02998 ± 0.00512
traded_value_cr               0.01920 ± 0.00966
vix_realized_spread           0.01318 ± 0.00654
dii_fut_net_pct               0.01149 ± 0.00494
vix_close                     0.01122 ± 0.01014
fii_opt_skew_pct              0.01095 ± 0.00253
pro_vol_fut_net_pct           0.00587 ± 0.00715  ⚠ not distinguishable from 0
basis                         0.00340 ± 0.00549  ⚠ not distinguishable from 0
pcr_chg_5d                    0.00180 ± 0.00307  ⚠ not distinguishable from 0
num_trades                    0.00121 ± 0.00083
basis_chg_5d                  0.00075 ± 0.00043
fut

In [84]:
results = con.execute("""
    SELECT MIN(trade_date), MAX(trade_date), COUNT(*)
    FROM nse.fii_stats
    WHERE instrument = 'INDEX FUTURES'
""").fetchall()
print("fii_stats INDEX FUTURES coverage:", results)

results = con.execute("""
    SELECT strftime(trade_date, '%Y') AS yr, COUNT(*) AS days
    FROM nse.fii_stats
    WHERE instrument = 'INDEX FUTURES'
    GROUP BY 1 ORDER BY 1
""").fetchall()
print("\nBy year:")
for row in results: print(row)

fii_stats INDEX FUTURES coverage: [(datetime.date(2016, 1, 1), datetime.date(2026, 7, 24), 2616)]

By year:
('2016', 247)
('2017', 248)
('2018', 246)
('2019', 245)
('2020', 252)
('2021', 248)
('2022', 248)
('2023', 246)
('2024', 249)
('2025', 249)
('2026', 138)


In [85]:
results = con.execute("""
    WITH ranked AS (
        SELECT
            NTILE(5) OVER (ORDER BY market_cap_cr) AS mcap_quintile,
            market_cap_cr, fwd_ret_1d, fwd_ret_5d, fwd_ret_20d, up_1d
        FROM daily_features
        WHERE fwd_ret_1d IS NOT NULL AND market_cap_cr IS NOT NULL
    )
    SELECT
        mcap_quintile,
        COUNT(*)                    AS days,
        ROUND(AVG(market_cap_cr), 0)   AS avg_mcap_cr,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
    FROM ranked
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("=== Market Cap Quintiles (full sample) ===")
for row in results: print(row)

results_train = con.execute("""
    WITH ranked AS (
        SELECT
            NTILE(5) OVER (ORDER BY market_cap_cr) AS mcap_quintile,
            market_cap_cr, fwd_ret_1d, fwd_ret_5d, fwd_ret_20d, up_1d
        FROM daily_features
        WHERE fwd_ret_1d IS NOT NULL AND market_cap_cr IS NOT NULL AND split = 'train'
    )
    SELECT
        mcap_quintile,
        COUNT(*)                    AS days,
        ROUND(AVG(market_cap_cr), 0)   AS avg_mcap_cr,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days
    FROM ranked
    GROUP BY 1
    ORDER BY 1
""").fetchall()

print("\n=== Market Cap Quintiles (TRAIN ONLY) ===")
for row in results_train: print(row)

=== Market Cap Quintiles (full sample) ===
(1, 523, 11198027.0, 0.062, 0.417, 2.177, 53.9)
(2, 523, 14466391.0, 0.041, 0.147, 0.857, 54.7)
(3, 523, 20624064.0, 0.096, 0.525, 1.357, 55.8)
(4, 523, 30746826.0, 0.038, 0.138, 0.532, 54.9)
(5, 522, 44881401.0, -0.002, -0.019, 0.085, 51.3)

=== Market Cap Quintiles (TRAIN ONLY) ===
(1, 469, 10952070.0, 0.072, 0.443, 2.106, 54.4)
(2, 469, 14195566.0, 0.039, 0.172, 1.461, 53.7)
(3, 469, 17714671.0, 0.07, 0.347, 0.523, 57.8)
(4, 469, 26756083.0, 0.036, 0.15, 0.518, 52.5)
(5, 469, 39857662.0, 0.054, 0.281, 1.15, 53.7)


In [86]:
mask_tail = df_train['max_pain_dist_pct'].abs() > 1.5
sub = df_train[mask_tail & df_train['fwd_ret_20d'].notna() & df_train['max_pain_dist_pct'].notna()]
ic, pval = stats.spearmanr(sub['max_pain_dist_pct'], sub['fwd_ret_20d'])
print(f"\nMax pain tail IC 20D (train only): {ic:.4f}, p={pval:.4f}, n={len(sub)}")


Max pain tail IC 20D (train only): -0.2712, p=0.0000, n=699


In [87]:
# First confirm metric_type values too, since we're assuming 'OI'
results = con.execute("""
    SELECT DISTINCT metric_type FROM nse.participant_activity ORDER BY 1
""").fetchall()
for row in results: print(row)

('OI',)
('VOL',)


In [88]:
results = con.execute("""
    SELECT trade_date, max_pain_dist_pct, fwd_ret_20d, split, vix_close
    FROM daily_features
    WHERE ABS(max_pain_dist_pct) > 1.5
      AND fwd_ret_20d IS NOT NULL
    ORDER BY trade_date
""").fetchall()
for row in results: print(row)

(datetime.date(2016, 1, 4), -1.7083, -4.3093, 'train', 16.835)
(datetime.date(2016, 1, 5), -1.5718, -5.4318, 'train', 16.7025)
(datetime.date(2016, 1, 6), -2.13, -4.3534, 'train', 16.5525)
(datetime.date(2016, 1, 7), -3.3891, -1.0465, 'train', 18.96)
(datetime.date(2016, 1, 8), -2.2622, -2.8166, 'train', 17.7975)
(datetime.date(2016, 1, 11), -2.1387, -3.5121, 'train', 18.69)
(datetime.date(2016, 1, 12), -2.8341, -3.9226, 'train', 18.7125)
(datetime.date(2016, 1, 13), -2.1812, -7.7495, 'train', 18.3525)
(datetime.date(2016, 1, 14), -2.3255, -7.3751, 'train', 18.605)
(datetime.date(2016, 1, 15), -3.1527, -3.6953, 'train', 19.415)
(datetime.date(2016, 1, 18), -3.8367, -4.1185, 'train', 20.225)
(datetime.date(2016, 1, 19), -2.5382, -4.3934, 'train', 18.43)
(datetime.date(2016, 1, 20), -3.1872, -1.6082, 'train', 20.965)
(datetime.date(2016, 1, 21), -3.384, -0.9077, 'train', 20.8125)
(datetime.date(2016, 2, 2), -1.5648, -3.1285, 'train', 18.035)
(datetime.date(2016, 2, 3), -2.7845, 0.0958, '

In [89]:
results = con.execute("""
    SELECT
        CASE
            WHEN vix_close < 14 THEN 'VIX_low'
            WHEN vix_close < 18 THEN 'VIX_mid'
            ELSE                     'VIX_high'
        END AS vix_regime,

        CASE
            WHEN basis < 0   THEN 'basis_discount'
            WHEN basis < 50  THEN 'basis_low'
            WHEN basis < 100 THEN 'basis_mid'
            ELSE                  'basis_high'
        END AS basis_regime,

        COUNT(*)                    AS days,
        ROUND(AVG(fwd_ret_1d), 3)   AS avg_ret_1d,
        ROUND(AVG(fwd_ret_5d), 3)   AS avg_ret_5d,
        ROUND(AVG(fwd_ret_20d), 3)  AS avg_ret_20d,
        ROUND(AVG(up_1d) * 100, 1)  AS pct_up_days

    FROM daily_features
    WHERE fwd_ret_1d IS NOT NULL
      AND split = 'train'
      AND vix_close IS NOT NULL
      AND basis IS NOT NULL

    GROUP BY 1, 2
    HAVING COUNT(*) >= 5
    ORDER BY 1, 2
""").fetchall()

print("=== VIX × Basis (TRAIN ONLY) ===")
for row in results: print(row)

=== VIX × Basis (TRAIN ONLY) ===
('VIX_high', 'basis_discount', 159, -0.107, 0.1, 2.117, 49.7)
('VIX_high', 'basis_high', 6, 0.066, 2.073, 4.417, 66.7)
('VIX_high', 'basis_low', 437, 0.133, 0.599, 2.536, 55.8)
('VIX_high', 'basis_mid', 61, 0.247, 0.647, 1.223, 57.4)
('VIX_low', 'basis_discount', 113, 0.08, 0.352, 0.605, 55.8)
('VIX_low', 'basis_high', 50, 0.042, -0.278, 0.614, 54.0)
('VIX_low', 'basis_low', 461, 0.033, 0.239, 0.981, 54.4)
('VIX_low', 'basis_mid', 165, 0.012, 0.119, 0.987, 57.0)
('VIX_mid', 'basis_discount', 94, 0.255, 0.514, 0.491, 66.0)
('VIX_mid', 'basis_high', 34, 0.117, 0.065, -0.226, 58.8)
('VIX_mid', 'basis_low', 630, 0.031, 0.208, 0.586, 53.3)
('VIX_mid', 'basis_mid', 134, -0.037, -0.09, 0.229, 44.8)


In [ ]:
con.close()